# Correct a set of row-summed spectra

This example reads closed-shutter and open-shutter polystyrene Zarr stores, holds out frames 0 and 42, and corrects both spectra together. Frame 0 is unchanged; the cosmic-ray event in frame 42 is corrected.

In [ ]:
import matplotlib.pyplot as plt

from racer_raman import SpectrumCalibrator, read_spectrum_zarr

In [ ]:
spectrum_stores = {
    "dark": "../data/spectrum/closed_shutter_1000ms_fvb.zarr",
    "polystyrene": "../data/spectrum/open_shutter_polystyrene_1000ms_fvb.zarr",
}
spectra = read_spectrum_zarr(spectrum_stores)

dark_frames = spectra.xs(("dark", 1000.0), level=("experiment", "exposure_ms"))
polystyrene_frames = spectra.xs(
    ("polystyrene", 1000.0), level=("experiment", "exposure_ms")
)
evaluation_frames = [0, 42]
reference_frames = polystyrene_frames.drop(index=evaluation_frames)
raw_spectra = polystyrene_frames.loc[evaluation_frames]
detector_columns = raw_spectra.columns.to_numpy()

In [ ]:
calibrator = SpectrumCalibrator.fit(dark_frames, reference_frames, exposure_ms=1000)
corrected_spectra = calibrator.correct_many(raw_spectra)

In [ ]:
measured_spectrum = raw_spectra.loc[42].to_numpy()
corrected_spectrum = corrected_spectra.loc[42].to_numpy()
fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True, constrained_layout=True)
axes[0].plot(detector_columns, measured_spectrum, label="measured spectrum")
axes[0].plot(detector_columns, corrected_spectrum, label="corrected spectrum")
axes[0].set_ylabel("ADU")
axes[0].legend()
axes[1].plot(detector_columns, measured_spectrum - corrected_spectrum, color="tab:red")
axes[1].set_xlabel("detector column")
axes[1].set_ylabel("removed ADU")
plt.show()

`corrected_spectra` keeps the input frame index, detector columns, and raw ADU scale. Spectra without an accepted cosmic-ray event remain unchanged.